# AR-order selection on paired simulated processes

This notebook compares BIC, PACF cutoff, first and second differences, and two Perron-vector rules on the retained width curves. Population widths also permit the original descent chain, local drift, cumulative-tail rules, and first tied minimum.

The simulation data and width-fitting settings are retained. Shared width rules are implemented in `ar_tools/breakpoint_selectors.py`; classical selectors are in `ar_tools/classical_selectors.py`.


In [ ]:
import hashlib
import itertools                                                                                     # infrastructure
import json                                                                                        # infrastructure
import sys                                                                                         # infrastructure
from pathlib import Path                                                                           # infrastructure
from time import perf_counter

import numpy as np
import pandas as pd
from scipy.optimize import minimize
from threadpoolctl import threadpool_limits                                                        # infrastructure

AR_CODE_DIR = next((                                                                               # infrastructure
    candidate.resolve()                                                                            # infrastructure
    for root in (Path.cwd(), *Path.cwd().parents)                                                  # infrastructure
    for candidate in (root / "Code" / "Code_AR_processes", root)                                 # infrastructure
    if (candidate / "ar_tools/functions.py").is_file()                                             # infrastructure
    and (candidate / "ar_tools/classical_selectors.py").is_file()                                             # infrastructure
), None)                                                                                           # infrastructure
if AR_CODE_DIR is None:                                                                            # infrastructure
    raise FileNotFoundError("Could not locate Code/Code_AR_processes.")                             # infrastructure
PROJECT_ROOT = AR_CODE_DIR.parents[1]                                                              # infrastructure
if str(PROJECT_ROOT) not in sys.path:                                                              # infrastructure
    sys.path.insert(0, str(PROJECT_ROOT))                                                          # infrastructure

from Code.Code_AR_processes.ar_tools.classical_selectors import (
    aic_bic_ar_order,
    pacf_cutoff_order,
)
from Code.Code_AR_processes.ar_tools.descent import batched_geometric_width_curves
from Code.Code_AR_processes.ar_tools.functions import (
    ar_impulse_response,
    companion_matrix,
    jones_ar_coefficients,
    simulate_uniform_ar,
)
from Code.Code_AR_processes.ar_tools.breakpoint_selectors import width_selectors
from Code.Code_AR_processes.ar3_population_breakpoint_selectors import all_population_selectors

NOTEBOOK_NAME = "ar_order_minimum_width"                                                          # infrastructure
RESULTS_DIR = AR_CODE_DIR / "outputs" / "csv" / NOTEBOOK_NAME                                    # infrastructure
RESULTS_DIR.mkdir(parents=True, exist_ok=True)                                                     # infrastructure

true_orders = tuple(range(1, 7))
settings = {
    "true_orders": list(true_orders),
    "replications_per_order": 50,
    "sample_size": 2_000,
    "burnin": 1_000,
    "half_width": 1.0,
    "k_values": list(range(1, 11)),
    "seeds": {
        "1": 20260813, "2": 20260814, "3": 20260816,
        "4": 20260817, "5": 20260818, "6": 20260819,
    },
    "starts": 3,
    "iterations": 500,
    "initial_step": 0.2,
    "geometric_decay": 0.99,
    "width_source_hashes": {                                                                      # infrastructure
        name: hashlib.sha256((AR_CODE_DIR / name).read_bytes()).hexdigest()                         # infrastructure
        for name in ("ar_tools/functions.py", "ar_tools/descent.py")                              # infrastructure
    },                                                                                             # infrastructure
}
k_values = np.asarray(settings["k_values"])

width_method_order = list(width_selectors([5, 4, 3, 1, 1, 1, 1, 1, 1, 1]))
method_order = ["BIC", "PACF cutoff"] + width_method_order
print("50 processes per true order; p=1,...,6; T=2,000; K=10.")


## Paired simulation

One deterministic random-number stream generates the coefficients and series for each replication. All retained selectors see matched data. BIC and PACF consider orders 0,...,9; geometric selectors see the same fitted width curve. A validated cache preserves the original curves and computes only missing fits.


In [ ]:
configuration_path = RESULTS_DIR / "configuration.json"                                # infrastructure
width_path = RESULTS_DIR / "width_curves.csv"                                                      # infrastructure
fixed_cache_keys = (                                                                               # infrastructure
    "replications_per_order", "sample_size", "burnin", "half_width",                            # infrastructure
    "k_values", "starts", "iterations", "initial_step",                                        # infrastructure
    "geometric_decay", "width_source_hashes",                                                    # infrastructure
)                                                                                                 # infrastructure
cached_settings = (                                                                               # infrastructure
    json.loads(configuration_path.read_text())                                                     # infrastructure
    if configuration_path.is_file() else {}                                                       # infrastructure
)                                                                                                 # infrastructure
cache_compatible = all(                                                                           # infrastructure
    cached_settings.get(key) == settings[key] for key in fixed_cache_keys                         # infrastructure
)                                                                                                 # infrastructure

width_columns = ["true_order", "replication", "k", "minimum_width", "coefficients"]             # infrastructure
if cache_compatible and width_path.is_file():                                                      # infrastructure
    cached_widths = pd.read_csv(width_path)                                                        # infrastructure
    cached_orders = set(cached_widths["true_order"].astype(int).unique())                          # infrastructure
    cache_compatible = all(                                                                        # infrastructure
        cached_settings.get("seeds", {}).get(str(order)) == settings["seeds"][str(order)]          # infrastructure
        for order in cached_orders & set(true_orders)                                              # infrastructure
    )                                                                                              # infrastructure
else:                                                                                              # infrastructure
    cached_widths = pd.DataFrame(columns=width_columns)                                            # infrastructure

valid_groups = []                                                                                  # infrastructure
if cache_compatible:                                                                              # infrastructure
    cached_widths = cached_widths[                                                                 # infrastructure
        cached_widths["true_order"].isin(true_orders)                                             # infrastructure
        & cached_widths["replication"].between(1, settings["replications_per_order"])              # infrastructure
    ]                                                                                              # infrastructure
    required_k = set(k_values.tolist())                                                            # infrastructure
    for _, group in cached_widths.groupby(["true_order", "replication"]):                         # infrastructure
        if (                                                                                       # infrastructure
            len(group) == len(k_values)                                                            # infrastructure
            and set(group["k"].astype(int)) == required_k                                         # infrastructure
            and not group["k"].duplicated().any()                                                 # infrastructure
            and np.isfinite(group["minimum_width"]).all()                                        # infrastructure
        ):                                                                                         # infrastructure
            valid_groups.append(group[width_columns])                                              # infrastructure
width_curves = (                                                                                   # infrastructure
    pd.concat(valid_groups, ignore_index=True)                                                     # infrastructure
    if valid_groups else pd.DataFrame(columns=width_columns)                                       # infrastructure
)                                                                                                 # infrastructure

expected_processes = {                                                                            # infrastructure
    (true_order, replication)                                                                     # infrastructure
    for true_order in true_orders                                                                 # infrastructure
    for replication in range(1, settings["replications_per_order"] + 1)                           # infrastructure
}                                                                                                 # infrastructure
cached_processes = set(map(tuple, width_curves[["true_order", "replication"]].astype(int).drop_duplicates().to_numpy()))  # infrastructure
missing_processes = sorted(expected_processes - cached_processes)                                  # infrastructure

if missing_processes:
    width_rows = width_curves.to_dict("records")
    started = perf_counter()
    with threadpool_limits(limits=1):                                                              # infrastructure
        for true_order in true_orders:
            experiment_seed = settings["seeds"][str(true_order)]
            missing_replications = [
                replication for order, replication in missing_processes
                if order == true_order
            ]
            for completed, replication in enumerate(missing_replications, start=1):
                replication_index = replication - 1
                generator = np.random.default_rng(np.random.SeedSequence([
                    experiment_seed, replication_index, 0,
                ]))
                coefficients = jones_ar_coefficients(true_order, rng=generator)
                series = simulate_uniform_ar(
                    coefficients,
                    sample_size=settings["sample_size"],
                    burnin=settings["burnin"],
                    half_width=settings["half_width"],
                    rng=generator,
                )
                _, minimum_widths, _ = batched_geometric_width_curves(
                    series,
                    int(k_values.max()),
                    seed=experiment_seed,
                    replication=replication_index,
                    starts=settings["starts"],
                    iterations=settings["iterations"],
                    initial_step=settings["initial_step"],
                    geometric_decay=settings["geometric_decay"],
                )
                for k, width in enumerate(minimum_widths, start=1):
                    width_rows.append({
                        "true_order": true_order,
                        "replication": replication,
                        "k": int(k),
                        "minimum_width": float(width),
                        "coefficients": json.dumps(coefficients.tolist()),
                    })
                if completed % 10 == 0:
                    print(
                        f"Completed {completed}/{len(missing_replications)} "
                        f"new AR({true_order}) curves"
                    )
    width_curves = (
        pd.DataFrame(width_rows)[width_columns]
        .sort_values(["true_order", "replication", "k"])
        .reset_index(drop=True)
    )
    print(
        f"Generated {len(missing_processes)} missing width curves "
        f"in {perf_counter() - started:.1f} seconds."
    )
else:
    print(f"Reused all {len(expected_processes)} cached width curves.")

width_curves.to_csv(width_path, index=False)                                                       # infrastructure
configuration_path.write_text(json.dumps(settings, indent=2) + "\n")                             # infrastructure
expected_width_rows = len(expected_processes) * len(k_values)                                     # infrastructure
assert len(width_curves) == expected_width_rows
assert set(map(tuple, width_curves[["true_order", "replication"]].astype(int).drop_duplicates().to_numpy())) == expected_processes


In [ ]:
selection_rows = []

for (true_order, replication), curve in width_curves.groupby(
    ["true_order", "replication"], sort=True
):
    true_order = int(true_order)
    replication = int(replication)
    experiment_seed = settings["seeds"][str(true_order)]

    # Regenerate the matched process used for this cached width curve.
    generator = np.random.default_rng(np.random.SeedSequence([
        experiment_seed, replication - 1, 0,
    ]))
    coefficients = jones_ar_coefficients(true_order, rng=generator)
    cached_coefficients = np.asarray(json.loads(curve.iloc[0]["coefficients"]))
    np.testing.assert_allclose(coefficients, cached_coefficients, rtol=0, atol=1e-14)
    series = simulate_uniform_ar(
        coefficients,
        sample_size=settings["sample_size"],
        burnin=settings["burnin"],
        half_width=settings["half_width"],
        rng=generator,
    )

    _, bic_order, _ = aic_bic_ar_order(series, maximum_order=9)
    pacf_order, _, _ = pacf_cutoff_order(series, maximum_order=9)
    selected_orders = {
        "BIC": bic_order,
        "PACF cutoff": pacf_order,
    }

    minimum_widths = curve.sort_values("k")["minimum_width"].to_numpy()
    selected_orders.update(width_selectors(minimum_widths))

    for method, selected_order in selected_orders.items():
        selection_rows.append({
            "true_order": true_order,
            "replication": replication,
            "method": method,
            "selected_order": int(selected_order),
        })

selections = pd.DataFrame(selection_rows)
selections.to_csv(RESULTS_DIR / "selections.csv", index=False)                                    # infrastructure

summary = (
    selections.assign(
        correct=lambda frame: frame["selected_order"] == frame["true_order"],
        under=lambda frame: frame["selected_order"] < frame["true_order"],
        over=lambda frame: frame["selected_order"] > frame["true_order"],
    )
    .groupby(["method", "true_order"], as_index=False)[["correct", "under", "over"]]
    .sum()
)
summary[["correct", "under", "over"]] = summary[["correct", "under", "over"]].astype(int)
summary["total"] = summary[["correct", "under", "over"]].sum(axis=1)
summary.to_csv(RESULTS_DIR / "summary.csv", index=False)                                           # infrastructure

assert (summary["total"] == 50).all()
assert len(selections) == len(true_orders) * 50 * len(method_order)

# These are the published BIC and PACF rows in Table 2.1.
table_2_1_reference = {
    ("BIC", 1): (47, 3, 0), ("BIC", 2): (46, 4, 0), ("BIC", 3): (43, 7, 0),
    ("PACF cutoff", 1): (31, 1, 18), ("PACF cutoff", 2): (34, 1, 15),
    ("PACF cutoff", 3): (33, 3, 14),
}
for (method, true_order), expected in table_2_1_reference.items():
    observed = summary.loc[
        (summary["method"] == method) & (summary["true_order"] == true_order),
        ["correct", "under", "over"],
    ].iloc[0]
    assert tuple(observed) == expected

print("Evaluated all retained selectors for AR(1)–AR(6); the original BIC and PACF columns still match Table 2.1.")


## Selection table

Each cell reports **correct selections / 50**, followed by **(underselections / overselections)**.


In [ ]:
summary_lookup = summary.set_index(["method", "true_order"])
comparison_table = pd.DataFrame(index=pd.Index(method_order, name="Selector"))
for true_order in true_orders:
    comparison_table[f"AR({true_order})"] = [
        f"{int(summary_lookup.loc[(method, true_order), 'correct'])}/50 "
        f"({int(summary_lookup.loc[(method, true_order), 'under'])}/"
        f"{int(summary_lookup.loc[(method, true_order), 'over'])})"
        for method in method_order
    ]
comparison_table.to_csv(RESULTS_DIR / "table_2_1_comparison.csv")
display(comparison_table)


The table reports the retained selectors on identical simulated processes. It does not change the stored optimizer outputs.


## Diagnostic using population minimum widths

This comparison removes the empirical width-fitting step from the selector input. The support-width formula and plateau follow Proposition `prop:minimum_width` in the thesis source. Pre-plateau values below are numerical approximations; the dedicated `ar3_population_breakpoint_selectors.ipynb` provides the more accurate AR(3) geometry audit.


In [ ]:
def _population_generator_matrix(impulse_response, dimension):
    """Rows are g_j=(psi_j, psi_{j-1}, ..., psi_{j-k+1})."""
    padded = np.r_[np.zeros(dimension - 1), impulse_response]
    return np.column_stack([
        padded[dimension - 1 - lag:dimension - 1 - lag + len(impulse_response)]
        for lag in range(dimension)
    ])


def _normalized_rows(vectors):
    vectors = np.asarray(vectors, dtype=float)
    norms = np.linalg.norm(vectors, axis=1)
    return vectors[norms > 1e-12] / norms[norms > 1e-12, None]


def _pole_factor_starts(coefficients, dimension):
    if dimension == 1:
        return []
    poles = np.linalg.eigvals(companion_matrix(coefficients))
    starts = []
    for indices in itertools.combinations(range(len(coefficients)), dimension - 1):
        factor = np.poly(poles[list(indices)])
        if np.max(np.abs(factor.imag)) < 1e-8:
            starts.append(factor.real / np.linalg.norm(factor.real))
    return starts


def _smooth_population_refinement(generator_matrix, start):
    direction = np.asarray(start, dtype=float)
    direction /= np.linalg.norm(direction)
    scale = max(float(np.max(np.linalg.norm(generator_matrix, axis=1))), 1.0)

    for epsilon_factor in (1e-3, 1e-5, 1e-7):
        epsilon = epsilon_factor * scale

        def objective_and_gradient(vector):
            vector_norm = np.linalg.norm(vector)
            unit_vector = vector / vector_norm
            projections = generator_matrix @ unit_vector
            smoothed_absolute = np.sqrt(projections**2 + epsilon**2)
            objective = 2.0 * np.sum(smoothed_absolute - epsilon)
            sphere_gradient = 2.0 * (
                generator_matrix.T @ (projections / smoothed_absolute)
            )
            gradient = (
                sphere_gradient
                - unit_vector * (unit_vector @ sphere_gradient)
            ) / vector_norm
            return objective, gradient

        result = minimize(
            objective_and_gradient,
            direction,
            method="L-BFGS-B",
            jac=True,
            options={
                "maxiter": 120, "ftol": 1e-13,
                "gtol": 1e-9, "maxls": 40,
            },
        )
        direction = result.x / np.linalg.norm(result.x)
    return direction


def _facet_normal_refinement(generator_matrix, directions, nearest=12):
    row_norms = np.linalg.norm(generator_matrix, axis=1)
    active = row_norms > 1e-11 * row_norms.max()
    rows = generator_matrix[active]
    unit_rows = rows / row_norms[active, None]
    candidates = [direction / np.linalg.norm(direction) for direction in directions]

    for direction in directions:
        distances = np.abs(unit_rows @ direction)
        count = min(nearest, len(rows))
        nearest_rows = rows[np.argpartition(distances, count - 1)[:count]]
        for indices in itertools.combinations(
            range(len(nearest_rows)), generator_matrix.shape[1] - 1
        ):
            _, singular_values, right_vectors = np.linalg.svd(
                nearest_rows[list(indices)], full_matrices=True
            )
            if (
                len(singular_values)
                and singular_values[-1] < 1e-10 * singular_values[0]
            ):
                continue
            normal = right_vectors[-1]
            normal_norm = np.linalg.norm(normal)
            if normal_norm > 1e-12:
                candidates.append(normal / normal_norm)

    candidate_matrix = np.asarray(candidates)
    widths = 2.0 * np.sum(
        np.abs(generator_matrix @ candidate_matrix.T), axis=0
    )
    best_indices = np.argsort(widths)[:8]
    return [candidate_matrix[index] for index in best_indices]


def approximate_population_width_curve(coefficients, seed):
    """Approximate m_k for k<=p and use the exact plateau for k>=p+1."""
    coefficients = np.asarray(coefficients, dtype=float)
    order = len(coefficients)
    spectral_radius = float(max(abs(np.linalg.eigvals(companion_matrix(coefficients)))))
    evaluation_history = min(
        250_000,
        max(20_000, int(np.ceil(np.log(1e-10) / np.log(spectral_radius)))),
    )
    optimization_history = min(3_000, evaluation_history)
    evaluation_impulse = ar_impulse_response(coefficients, evaluation_history)
    optimization_impulse = evaluation_impulse[:optimization_history + 1]
    plateau = 2.0 / np.linalg.norm(np.r_[1.0, -coefficients])
    generator = np.random.default_rng(seed)

    widths = []
    directions = []
    half_history_increments = []
    previous_direction = None

    for dimension in range(1, order + 1):
        optimization_generators = _population_generator_matrix(
            optimization_impulse, dimension
        )
        evaluation_generators = _population_generator_matrix(
            evaluation_impulse, dimension
        )

        starts = []
        if previous_direction is not None:
            starts.append(np.r_[previous_direction, 0.0])
        starts.extend(np.eye(dimension))
        starts.append(np.r_[1.0, -coefficients[:dimension - 1]])
        starts.extend(_pole_factor_starts(coefficients, dimension))
        starts.extend(_normalized_rows(generator.normal(size=(48, dimension))))
        starts = _normalized_rows(starts)

        start_widths = 2.0 * np.sum(
            np.abs(optimization_generators @ starts.T), axis=0
        )
        best_starts = starts[np.argsort(start_widths)[:8]]
        smooth_directions = [
            _smooth_population_refinement(optimization_generators, start)
            for start in best_starts
        ]
        candidate_directions = _facet_normal_refinement(
            optimization_generators, smooth_directions, nearest=12
        ) + smooth_directions
        candidate_matrix = np.asarray(candidate_directions)
        evaluation_widths = 2.0 * np.sum(
            np.abs(evaluation_generators @ candidate_matrix.T), axis=0
        )
        best_index = int(np.argmin(evaluation_widths))
        direction = candidate_matrix[best_index]
        width = float(evaluation_widths[best_index])
        half_width = float(2.0 * np.sum(np.abs(
            evaluation_generators[:len(evaluation_generators) // 2] @ direction
        )))

        widths.append(width)
        directions.append(direction)
        half_history_increments.append(width - half_width)
        previous_direction = direction

    widths.extend([plateau] * (len(k_values) - order))
    return {
        "widths": np.asarray(widths),
        "directions": directions,
        "half_history_increments": np.asarray(half_history_increments),
        "evaluation_history": evaluation_history,
        "spectral_radius": spectral_radius,
        "plateau": plateau,
    }


In [ ]:
population_width_path = RESULTS_DIR / "population_width_curves.csv"                     # infrastructure
population_configuration_path = (                                                               # infrastructure
    RESULTS_DIR / "population_configuration.json"                                                # infrastructure
)                                                                                                 # infrastructure
process_parameters = (                                                                            # infrastructure
    width_curves[["true_order", "replication", "coefficients"]]                                  # infrastructure
    .drop_duplicates()                                                                             # infrastructure
    .sort_values(["true_order", "replication"])                                                   # infrastructure
    .reset_index(drop=True)                                                                        # infrastructure
)                                                                                                 # infrastructure
coefficient_source_hash = hashlib.sha256(                                                         # infrastructure
    process_parameters.to_csv(index=False).encode()                                                # infrastructure
).hexdigest()                                                                                      # infrastructure
population_settings = {                                                                           # infrastructure
    "method_version": 1,                                                                          # infrastructure
    "coefficient_source_hash": coefficient_source_hash,                                           # infrastructure
    "optimization_history": 3_000,                                                               # infrastructure
    "minimum_evaluation_history": 20_000,                                                        # infrastructure
    "maximum_evaluation_history": 250_000,                                                       # infrastructure
    "spectral_tail_target": 1e-10,                                                               # infrastructure
    "random_starts": 48,                                                                          # infrastructure
    "refined_starts": 8,                                                                          # infrastructure
}                                                                                                 # infrastructure
expected_population_rows = len(true_orders) * 50 * len(k_values)                                 # infrastructure
use_population_cache = (                                                                          # infrastructure
    population_width_path.is_file()                                                               # infrastructure
    and population_configuration_path.is_file()                                                   # infrastructure
    and json.loads(population_configuration_path.read_text()) == population_settings               # infrastructure
)                                                                                                 # infrastructure
if use_population_cache:                                                                          # infrastructure
    population_widths = pd.read_csv(population_width_path)                                        # infrastructure
    use_population_cache = (                                                                      # infrastructure
        len(population_widths) == expected_population_rows                                        # infrastructure
        and not population_widths.duplicated(["true_order", "replication", "k"]).any()            # infrastructure
        and np.isfinite(population_widths["population_minimum_width"]).all()                       # infrastructure
    )                                                                                              # infrastructure

if not use_population_cache:
    population_rows = []
    started = perf_counter()
    with threadpool_limits(limits=1):                                                              # infrastructure
        for process in process_parameters.itertuples(index=False):
            true_order = int(process.true_order)
            replication = int(process.replication)
            coefficients = np.asarray(json.loads(process.coefficients))
            approximation = approximate_population_width_curve(
                coefficients,
                seed=[20260922, true_order, replication],
            )
            for k, width in zip(k_values, approximation["widths"]):
                is_approximation = k <= true_order
                population_rows.append({
                    "true_order": true_order,
                    "replication": replication,
                    "k": int(k),
                    "population_minimum_width": float(width),
                    "method": (
                        "numerical approximation"
                        if is_approximation else "exact plateau formula"
                    ),
                    "history": (
                        approximation["evaluation_history"]
                        if is_approximation else 0
                    ),
                    "half_history_increment": (
                        float(approximation["half_history_increments"][k - 1])
                        if is_approximation else 0.0
                    ),
                    "plateau": approximation["plateau"],
                })
            if replication % 10 == 0:
                print(
                    f"Completed {replication}/50 AR({true_order}) "
                    "population curves"
                )
    population_widths = pd.DataFrame(population_rows)
    population_widths.to_csv(population_width_path, index=False)                                  # infrastructure
    population_configuration_path.write_text(                                                      # infrastructure
        json.dumps(population_settings, indent=2) + "\n"                                         # infrastructure
    )                                                                                              # infrastructure
    print(
        f"Generated 300 population curves in "
        f"{perf_counter() - started:.1f} seconds."
    )
else:
    print("Reused all 300 cached population-width curves.")

# Numerical quality checks.
population_groups = population_widths.groupby(["true_order", "replication"])
monotonicity_violations = sum(
    np.any(np.diff(group.sort_values("k")["population_minimum_width"]) > 1e-7)
    for _, group in population_groups
)
pre_plateau_violations = sum(
    np.any(
        group[group["k"] <= true_order]["population_minimum_width"]
        <= group["plateau"].iloc[0]
    )
    for (true_order, _), group in population_groups
)
relative_half_history_increment = (
    population_widths["half_history_increment"].abs()
    / population_widths["population_minimum_width"].clip(lower=1e-300)
)
assert monotonicity_violations == 0
assert pre_plateau_violations == 0
assert relative_half_history_increment.max() < 1.1e-5

# Repeat the hardest process at each order with an independent random-start seed.
radius_rows = []
for process in process_parameters.itertuples(index=False):
    coefficients = np.asarray(json.loads(process.coefficients))
    radius_rows.append({
        "true_order": int(process.true_order),
        "replication": int(process.replication),
        "coefficients": coefficients,
        "spectral_radius": float(max(abs(np.linalg.eigvals(companion_matrix(coefficients))))),
    })
radius_table = pd.DataFrame(radius_rows)
validation_differences = []
with threadpool_limits(limits=1):                                                                  # infrastructure
    for _, process in radius_table.sort_values(
        "spectral_radius", ascending=False
    ).groupby("true_order", sort=True).head(1).iterrows():
        repeated = approximate_population_width_curve(
            process["coefficients"],
            seed=[20260923, int(process["true_order"]), int(process["replication"])],
        )["widths"]
        cached = population_widths.loc[
            (population_widths["true_order"] == process["true_order"])
            & (population_widths["replication"] == process["replication"])
        ].sort_values("k")["population_minimum_width"].to_numpy()
        validation_differences.append(
            float(np.max(np.abs(repeated - cached) / np.maximum(cached, 1e-300)))
        )
assert max(validation_differences) < 1e-7

print(
    "Approximation checks: no monotonicity or plateau-bound violations; "
    f"maximum half-history increment {relative_half_history_increment.max():.2e}; "
    f"maximum independent-restart change {max(validation_differences):.2e}."
)


### Population-width selection table

Each cell reports **correct / 50 (under / over)**.


In [ ]:
population_selection_rows = []
for (true_order, replication), curve in population_widths.groupby(["true_order", "replication"], sort=True):
    m = curve.sort_values("k").population_minimum_width.to_numpy()
    for method, selected_order in all_population_selectors(m).items():
        population_selection_rows.append({"true_order": int(true_order), "replication": int(replication),
                                          "method": method, "selected_order": selected_order})
population_selections = pd.DataFrame(population_selection_rows)
population_selections.to_csv(RESULTS_DIR / "population_selections.csv", index=False)
population_summary = population_selections.assign(
    correct=lambda f: f.selected_order == f.true_order,
    under=lambda f: f.selected_order < f.true_order,
    over=lambda f: f.selected_order > f.true_order,
).groupby(["method", "true_order"], as_index=False)[["correct", "under", "over"]].sum()
population_summary["total"] = population_summary[["correct", "under", "over"]].sum(axis=1)
population_summary.to_csv(RESULTS_DIR / "population_summary.csv", index=False)
population_table = population_summary.pivot(index="method", columns="true_order", values="correct")
population_table.to_csv(RESULTS_DIR / "population_width_selector_table.csv")
display(population_table)


In [ ]:
attribution = selections.merge(population_selections,
    on=["method", "true_order", "replication"], suffixes=("_empirical", "_population"))
attribution["empirical_correct"] = attribution.selected_order_empirical == attribution.true_order
attribution["population_correct"] = attribution.selected_order_population == attribution.true_order
attribution_table = attribution.groupby("method").agg(
    empirical_correct=("empirical_correct", "sum"), population_correct=("population_correct", "sum"),
    total=("replication", "count"))
attribution_table.to_csv(RESULTS_DIR / "empirical_population_attribution.csv")
display(attribution_table)
